In [ ]:
import os
import asyncio
import re
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode
from crawl4ai.async_dispatcher import SemaphoreDispatcher

# The exact base URL structure
BASE_LAW_URL = "https://legalinfo.mn/mn/law?page=law&cate={cate_id}&active=1&page={page_num}&sort=title"
DETAIL_URL = "https://legalinfo.mn/mn/detail?lawId="

categories = {
    "Монгол Улсын Үндсэн Хууль": 26,
    "Монгол Улсын хууль": 27,
    "Улсын Их Хурлын тогтоол": 28,
    "Монгол Улсын олон улсын гэрээ": 29,
    "Ерөнхийлөгчийн зарлиг": 30,
    "Үндсэн хуулийн цэцийн шийдвэр": 31,
    "Улсын дээд шүүхийн тогтоол": 32,
    "Шүүхийн ерөнхий зөвлөл": 16231124857801,
    "Засгийн газрын тогтоол": 33,
    "Сайдын тушаал": 34,
    "Засгийн газрын агентлагийн даргын тушаал": 35,
    "УИХ-аас томилогддог байгууллагын дарга, түүнтэй адилтгах албан тушаалтны шийдвэр": 36,
    "Аймаг, нийслэлийн ИТХ-ын шийдвэр": 37,
    "Аймаг, нийслэлийн Засаг даргын захирамж": 38,
    "Төрийн зарим чиг үүргийг хууль болон гэрээний үндсэн дээр хэрэгжүүлж буй байгууллага": 180,
    "Зөвлөл, хороо, бусад байгууллага": 186,
    "Хууль, хяналтын байгууллага": 360
}

async def crawl_legal_system():
    # Use a semaphore to avoid overloading the server
    dispatcher = SemaphoreDispatcher(max_session_permit=30) 
    
    async with AsyncWebCrawler() as crawler:
        for cat_name, cat_id in categories.items():
            # Create a clean folder name
            folder_name = re.sub(r'[\\/*?:"<>|]', "", cat_name).strip()
            os.makedirs(folder_name, exist_ok=True)
            
            page_num = 1
            while True:
                list_url = BASE_LAW_URL.format(cate_id=cat_id, page_num=page_num)
                print(f"\n--- Category: {cat_name} | Page: {page_num} ---")
                
                # 1. Fetch with specific wait condition for the law body class
                result = await crawler.arun(
                    url=list_url, 
                    config=CrawlerRunConfig(
                        cache_mode=CacheMode.BYPASS,
                        # Wait for the specific container to ensure JS has rendered the list
                        wait_for="css:.shine-huuli-body.mt-0" 
                    )
                )

                # 2. Robust ID Extraction
                # We check both the internal links list AND the raw HTML
                law_ids = set()
                
                # Method A: Crawl4AI's parsed internal links
                for link in result.links.get("internal", []):
                    href = link.get('href', '')
                    match = re.search(r'lawId=(\d+)', href)
                    if match:
                        law_ids.add(match.group(1))
                print(f"There are {len(law_ids)} laws in page number {page_num}")
                # Method B: Regex fallback on raw HTML if links object is empty
                if not law_ids:
                    law_ids = set(re.findall(r'lawId=(\d+)', result.html))

                if not law_ids:
                    print(f"Finished category: {cat_name} (No laws found on page {page_num})")
                    break 
                
                print(f"Found {len(law_ids)} laws on page {page_num}")

                # 3. Process the laws
                for law_id in law_ids:
                    file_path = os.path.join(folder_name, f"{law_id}.md")
                    
                    # Check if we already have it
                    if not os.path.exists(file_path):
                        detail_url = f"{DETAIL_URL}{law_id}"
                        law_data = await crawler.arun(
                            url=detail_url,
                            config=CrawlerRunConfig(cache_mode=CacheMode.BYPASS)
                        )
                        
                        if law_data.success:
                            with open(file_path, "w", encoding="utf-8") as f:
                                f.write(law_data.markdown)
                            print(f"  [+] Saved {law_id}")
                        else:
                            print(f"  [!] Failed to download {law_id}")
                    
                page_num += 1
                await asyncio.sleep(0.2) # Maintain a polite crawl speed

if __name__ == "__main__":
    # If running in a script/jupyter
    await crawl_legal_system()

/home/amon/miniconda3/envs/research/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


[INIT].... → Crawl4AI 0.8.0 


--- Category: Улсын Их Хурлын тогтоол | Page: 1 ---


[FETCH]... ↓ https://legalinfo.mn/mn/law?page=law&cate=28&active=1&page=1&sort=title                              |
✓ | ⏱: 4.44s 

[SCRAPE].. ◆ https://legalinfo.mn/mn/law?page=law&cate=28&active=1&page=1&sort=title                              |
✓ | ⏱: 0.19s 

...

  [!] Failed to download 11355


[ERROR]... × https://legalinfo.mn/mn...il?lawId=16531127324971  | Error: Unexpected error in _crawl_web at line 718
in _crawl_web (../miniconda3/envs/research/lib/python3.10/site-packages/crawl4ai/async_crawler_strategy.py):
Error: Failed on navigating ACS-GOTO:
Page.goto: net::ERR_INTERNET_DISCONNECTED at https://legalinfo.mn/mn/detail?lawId=16531127324971
Call log:
  - navigating to "https://legalinfo.mn/mn/detail?lawId=16531127324971", waiting until "domcontentloaded"


Code context:
 713                                   tag="GOTO",
 714                                   params={"url": url},
 715                               )
 716                               response = None
 717                           else:
 718 →                             raise RuntimeError(f"Failed on navigating ACS-GOTO:\n{str(e)}")
 719   
 720                       # ──────────────────────────────────────────────────────────────
 721                       # Walk the redirect chain.  Playwright returns only the last
 722                       # hop, so we trace the `request.redirected_from` links until the
 723                       # first response that differs from the final one and surface its 

  [!] Failed to download 16531127324971

--- Category: Улсын Их Хурлын тогтоол | Page: 12 ---


[ERROR]... × https://legalinfo.mn/mn...ve=1&page=12&sort=title  | Error: Unexpected error in _crawl_web at line 718
in _crawl_web (../miniconda3/envs/research/lib/python3.10/site-packages/crawl4ai/async_crawler_strategy.py):
Error: Failed on navigating ACS-GOTO:
Page.goto: net::ERR_INTERNET_DISCONNECTED at 
https://legalinfo.mn/mn/law?page=law&cate=28&active=1&page=12&sort=title
Call log:
  - navigating to "https://legalinfo.mn/mn/law?page=law&cate=28&active=1&page=12&sort=title", waiting until 
"domcontentloaded"


Code context:
 713                                   tag="GOTO",
 714                                   params={"url": url},
 715                               )
 716                               response = None
 717                           else:
 718 →                             raise RuntimeError(f"Failed on navigating ACS-GOTO:\n{str(e)}")
 719   
 720                       # ──────────────────────────────────────────────────────────────
 721                       # Walk the redirect chain.  Playwright returns only the last
 722                       # hop, so we trace the `request.redirected_from` links until the
 723                       # first response that differs from the final one and surface its 

There are 0 laws in page number 12
Finished category: Улсын Их Хурлын тогтоол (No laws found on page 12)
